# Supply Chain Analysis
## Beauty Products Supply Chain Data Analysis

**Objective:** Identify risk patterns, efficiency and profitability
across a 100-product supply chain.

**Dataset:** Supply Chain Analysis — Kaggle
**Tools:** Python, Pandas
**Author:** Raul Farias

## 1. Data Loading and Exploration
Loading the dataset and checking data quality before analysis.

In [3]:
# Data Loading and Initial Exploration
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/supply_chain_data.csv')

print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nNull values:\n{df.isna().sum()}")
print(f"\nData types:\n{df.dtypes}")
df.head()

Mounted at /content/drive
Dataset shape: 100 rows, 24 columns

Null values:
Product type               0
SKU                        0
Price                      0
Availability               0
Number of products sold    0
Revenue generated          0
Customer demographics      0
Stock levels               0
Lead times                 0
Order quantities           0
Shipping times             0
Shipping carriers          0
Shipping costs             0
Supplier name              0
Location                   0
Lead time                  0
Production volumes         0
Manufacturing lead time    0
Manufacturing costs        0
Inspection results         0
Defect rates               0
Transportation modes       0
Routes                     0
Costs                      0
dtype: int64

Data types:
Product type                object
SKU                         object
Price                      float64
Availability                 int64
Number of products sold      int64
Revenue generated          

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632


## 2. Revenue Analysis by Product Type
Identifying which product category generates the most revenue.

In [5]:
group=df.groupby('Product type')['Revenue generated'].sum().round(2)
total_revenue=group.sum()
best_type=group.idxmax()
best_revenue=group.max()
percentage=((best_revenue/total_revenue)*100).round(1)
group=group.sort_values(ascending=False)
print(group)

Product type
skincare     241628.16
haircare     174455.39
cosmetics    161521.27
Name: Revenue generated, dtype: float64


### Conclusion
**Skincare** leads revenue generation with **$241,628** — representing **41.8%** of total revenue.

**Skincare** should be prioritized for inventory management and supplier negotiations.

## 3. Critical Stock Analysis
Identifying products with high sales volume but critically low stock levels.

In [ ]:
# Critical Stock Analysis
# Criteria: Stock levels < 10 AND Number of products sold > 500
filter=df[(df['Stock levels']<10) & (df['Number of products sold']>500)]
result=filter[['SKU','Product type','Stock levels','Number of products sold','Revenue generated']]
order=result.sort_values(by='Stock levels', ascending=False)
print(f'{order} \n Critical products found: {len(result)}')

      SKU Product type  Stock levels  Number of products sold  \
4    SKU4     skincare             5                      871   
78  SKU78     haircare             5                      946   
33  SKU33    cosmetics             4                      616   
47  SKU47     skincare             4                      910   
34  SKU34     skincare             1                      602   

    Revenue generated  
4         2686.505152  
78        1292.458418  
33        5149.998350  
47        7089.474250  
34        9061.710896   
 Critical products found: 5


### Conclusion
5 products show critical inventory risk — high sales volume with stock levels below 10 units.

**Most urgent case:** SKU34 (skincare) with only 1 unit in stock and 602 units sold —
immediate replenishment is required to avoid stockout.

**Recommendation:** Implement automatic reorder alerts for products with
stock levels below 10 and sales above 500 units.


## 4. Defect Rate Analysis by Product Type
Identifying which product category has the highest quality issues.

In [ ]:
group=df.groupby('Product type')['Defect rates'].mean().round(2).sort_values(ascending=False)
category_defect_top=group.idxmax()
higher_defect=group.max()
print(group)


Product type
haircare     2.48
skincare     2.33
cosmetics    1.92
Name: Defect rates, dtype: float64


### Conclusion
**Haircare** shows the highest average defect rate at **2.48%**,
followed by Skincare (**2.33%**) and Cosmetics (**1.92%**).

**Recommendation:** Quality control processes should focus on Haircare products
to reduce defect rates and protect brand reputation.

## 5. Shipping Cost Efficiency by Route
Identify which route has the best cost efficiency according to revenue generated.

In [ ]:
group=df.groupby('Routes').agg(
    Revenue_total=('Revenue generated','sum'),
    Shipping_cost_total=('Shipping costs','sum'),
).round(2)
group['Efficiency_ratio']=((group['Shipping_cost_total']/group['Revenue_total'])*100).round(2)
order=group.sort_values(by='Efficiency_ratio',ascending=True)
print(order)

         Revenue_total  Shipping_cost_total  Efficiency_ratio
Routes                                                       
Route A      253198.85               231.33              0.09
Route B      204484.01               205.42              0.10
Route C      119921.96               118.06              0.10


### Conclusion
**Route A** is the most cost-efficient route, with only **0.09%** of revenue
consumed in shipping costs — the lowest ratio across all routes.

**Recommendation:** Prioritize Route A for high-value shipments to maximize
revenue retention and minimize logistics costs.

## Supplier Performance
To identify which is the supplier that has the best performance

In [ ]:
group=(df.groupby('Supplier name').agg(
    Revenue_total=('Revenue generated','sum'),
    Manufacturing_costs_total=('Manufacturing costs','sum')).round(2)
)
group['Efficiency_ratio']=(group['Manufacturing_costs_total']/group['Revenue_total']*100).round(2)
group['Classification']=group.apply(lambda x: 'Efficient' if x['Efficiency_ratio']<1 else 'Regular',axis=1)
sort=group.sort_values(by='Efficiency_ratio', ascending=True)
print(sort)

               Revenue_total  Manufacturing_costs_total  Efficiency_ratio  \
Supplier name                                                               
Supplier 3          97795.98                     654.51              0.67   
Supplier 2         125467.42                     915.70              0.73   
Supplier 5         110343.46                     805.83              0.73   
Supplier 1         157529.00                    1221.86              0.78   
Supplier 4          86468.96                    1128.78              1.31   

              Classification  
Supplier name                 
Supplier 3         Efficient  
Supplier 2         Efficient  
Supplier 5         Efficient  
Supplier 1         Efficient  
Supplier 4           Regular  


### Conclusion
**4 out of 5 suppliers** are classified as Efficient (cost ratio below 1% of revenue).

**Supplier 3** shows the best performance with only **0.67%** of revenue
consumed in manufacturing costs.

**Supplier 4** is the only Regular supplier with **1.31%** — nearly double
the average of the other suppliers.

**Recommendation:** Evaluate alternative suppliers for Supplier 4's products
to reduce manufacturing costs and improve overall margin.

## General Conclusions

This supply chain analysis of 100 beauty products revealed four key insights:

1. **Revenue:** Skincare dominates with $241,628 (41.8% of total revenue) —
   the primary category for business focus.

2. **Inventory Risk:** 5 products show critical stock levels (below 10 units)
   with high sales volume — immediate replenishment required, especially SKU34.

3. **Quality:** Haircare has the highest defect rate (2.48%) —
   quality control processes need improvement in this category.

4. **Logistics:** Route A is the most cost-efficient (0.09% of revenue in shipping costs) —
   recommended for high-value shipments.

5. **Suppliers:** 4 out of 5 suppliers are efficient. Supplier 4 requires
   attention with a cost ratio of 1.31% — nearly double the average.

**Overall Recommendation:** Focus inventory and quality efforts on Haircare and Skincare
categories while optimizing logistics through Route A and reviewing Supplier 4's contracts.